![QuantConnect Logo](https://cdn.quantconnect.com/web/i/icon.png)
<hr>

## Consolidator Warm-Up Research

Warm a consolidator and its registered indicators before scheduled trade checks.

### Set Up QuantBook

Create a minute SPY subscription for the consolidator and history replay.

In [ ]:
qb = QuantBook()
qb.set_start_date(2024, 9, 1)
qb.settings.seed_initial_prices = True
equity = qb.add_equity("SPY", Resolution.MINUTE)

### Build Consolidator Indicators

Create a 30-minute consolidator, register the EMA and RSI to it, then warm the full chain from one history request.

In [ ]:
# Build the 30-minute consolidator.
consolidator = TradeBarConsolidator(timedelta(minutes=30))
# Subscribe the consolidator for automatic updates.
qb.subscription_manager.add_consolidator(equity, consolidator)
# Create 30-minute indicators for the scheduled trading signal.
ema = ExponentialMovingAverage(10)
rsi = RelativeStrengthIndex(12)
indicators = [ema, rsi]
for indicator in indicators:
    qb.register_indicator(equity, indicator, consolidator)
# Warm the consolidator and registered indicators with one history request.
for bar in qb.history[TradeBar](equity, max(indicator.warm_up_period for indicator in indicators)):
    consolidator.update(bar)
    ema.update(bar)
    rsi.update(bar)

### Verify Readiness

Confirm both registered indicators have enough samples before the strategy evaluates scheduled trades.

In [ ]:
print(f"EMA samples: {ema.samples}")
print(f"EMA ready: {ema.is_ready}")
print(f"EMA value: {ema.current.value:.2f}")
print(f"RSI samples: {rsi.samples}")
print(f"RSI ready: {rsi.is_ready}")
print(f"RSI value: {rsi.current.value:.2f}")
assert all([indicator.samples >= indicator.warm_up_period for indicator in indicators])
assert all([indicator.is_ready for indicator in indicators])

### Build Time Series

Replay a longer minute history window and store raw EMA and RSI values.

In [ ]:
series_consolidator = TradeBarConsolidator(timedelta(minutes=30))
series_ema = ExponentialMovingAverage(10)
series_rsi = RelativeStrengthIndex(12)
series_indicators = [series_ema, series_rsi]
for indicator in series_indicators:
    qb.register_indicator(equity, indicator, series_consolidator)

value_rows = []
signal_rows = []
for bar in qb.history[TradeBar](equity, 390, Resolution.MINUTE):
    series_consolidator.update(bar)
    series_ema.update(bar)
    series_rsi.update(bar)
    if not all([indicator.is_ready for indicator in series_indicators]):
        continue
    value_rows.append({"time": bar.end_time, "ema": series_ema.current.value, "rsi": series_rsi.current.value})
    signal_rows.append({
        "time": bar.end_time,
        "signal": 1 if bar.close > series_ema.current.value and series_rsi.current.value > 50 else 0
    })

indicator_values = pd.DataFrame(value_rows).set_index("time")
indicator_values.tail()

### Signal Series

Display the 1/0 signal generated by the scheduled trade rule.

In [ ]:
signals = pd.DataFrame(signal_rows).set_index("time")
signals.tail()